# Volatility forecasting with path signatures

One experiment, run end to end: for each horizon (5 min, 1 hour, 1 / 7 / 30 day) every model is fit on the **same purged expanding-window walk-forward folds**, and every table and figure below is generated from the saved results.

- The heavy work lives in `experiment.py` (runner) and `reporting.py` (tables/figures); this notebook is only config, run, and display.
- **Purge:** the target is realized volatility over the next *h*, so training rows whose label window reaches the first test time are dropped from each fold.
- Each horizon runs in its own Python process and finished horizons are skipped on re-run, so a crash cannot take the kernel down and can be resumed.
- Results are written to `results/<run>/<horizon>/`. Re-run only the display cells to regenerate figures.

In [ ]:
import logging
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import experiment as ex
import reporting as rp

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%H:%M:%S")
pd.set_option("display.float_format", "{:.6g}".format)

## 1. Configuration

In [ ]:
QUICK = False  # True: small, fast run to check that everything works

cfg = ex.ExperimentConfig(
    horizons=("5m", "1h", "1d", "7d", "30d"),
    n_folds=3,                       # expanding-window folds (+1 train-only block)
    binance_start="2026-07-01",      # ~2.5 months of ticks, streamed one day at a time
    binance_end="2026-09-15",
    yahoo_start="2015-01-01",        # full Yahoo history, no cap
    target_rows={"5m": 4000, "1h": 4000, "1d": None, "7d": None, "30d": None},
    torch_epochs=200,
    run_ablations=True,
    output_dir="results/main",
)
if QUICK:
    cfg = cfg.quick()
cfg

## 2. Run

Binance days are downloaded once, reduced to 5-second and 1-minute bars, and cached under `data/bars/`; signature features are cached under `cache/`. Safe to re-run: completed horizons are skipped (delete `results/<run>/<horizon>/` to redo one).

In [ ]:
ex.run_all(cfg, subprocess_per_horizon=True)

## 3. Results

In [ ]:
res = rp.load_results(cfg.output_dir)
pd.DataFrame(res.meta).T[["n_rows", "stride", "horizon_steps", "horizon_rows", "n_folds", "first_time", "last_time", "seconds"]]

### Model ranking per horizon

RMSE averaged over folds, with the fold-to-fold SD. `Pct_vs_Mean` / `Pct_vs_HAR` are % RMSE improvements over the no-information and HAR-RV-L forecasts.

In [ ]:
summary = rp.summary_table(res)
display(summary.style.format({"Mean_RMSE": "{:.6g}", "Fold_SD": "{:.6g}", "Pct_vs_Mean": "{:.1f}", "Pct_vs_HAR": "{:.1f}"}))

### Diebold-Mariano tests (pooled out-of-sample)

A low p-value means that model is significantly different from the reference, not just numerically so. The test lag is the horizon length in forecast rows, since overlapping targets autocorrelate the errors.

In [ ]:
for hk in res.meta:
    print(f"\n{ex.HORIZON_LABELS[hk]}")
    display(rp.dm_table(res, hk).style.format(
        {"Pooled_RMSE": "{:.6g}", "DM_vs_best": "{:.2f}", "p_vs_best": "{:.3g}", "DM_vs_HAR": "{:.2f}", "p_vs_HAR": "{:.3g}"}))

## 4. Figures

### Improvement over the mean forecast

In [ ]:
rp.plot_relative_heatmap(res); plt.show()

### Fold stability

A model is more convincing if it beats the mean in every fold, not just on average. Values below 1 beat the mean.

In [ ]:
rp.plot_fold_stability(res); plt.show()

### Actual vs predicted volatility

Best model (lowest fold-mean RMSE) against HAR-RV-L.

In [ ]:
rp.plot_predictions(res); plt.show()

### When does the best model gain or lose against HAR?

Cumulative `HAR squared error - best-model squared error`; rising means the model is adding value.

In [ ]:
rp.plot_cumulative_advantage(res); plt.show()

### Calm vs stressed periods

Regimes are terciles of realized volatility in the out-of-sample target; descriptive only, never a model input.

In [ ]:
rp.plot_regimes(res); plt.show()

### What in the signature construction matters?

Ridge on signatures with the folds held fixed, so only the representation changes. Compared against a control in each panel: level 1 only, the raw path, price only. These are descriptive and are not used to pick the specification in the main table.

In [ ]:
rp.plot_ablations(res); plt.show()

## 5. Save summary

In [ ]:
summary.to_csv(f"{cfg.output_dir}/model_summary.csv", index=False)
print("saved", f"{cfg.output_dir}/model_summary.csv")